In [ ]:
import pandas as pd
import re
from spacy.lang.es import stop_words
import pickle
from wordcloud import WordCloud
import matplotlib.pyplot as plt

SPCONTRACTIONS_PATH = 'Verbatim Cleaning Dicts/SP/SPcontractions.txt'
SPSLANG_PATH = 'Verbatim Cleaning Dicts/SP/SPslang.txt'

# Cargamos diccionarios de slangs y contracciones comunes en español 
dict_contracciones = pd.read_csv(SPCONTRACTIONS_PATH, sep='\t', header=None, index_col=0).to_dict()[1]
dict_slang = pd.read_csv(SPSLANG_PATH, sep='\t', header=None, index_col=0).to_dict()[1]
spa_stopwords = stop_words.STOP_WORDS

# Definimos una función que reemplace texto por cada entrada en el diccionario, va palabra por palabra en un texto
def dict_replacer(text='', dictionary={}):
    for original, new in dictionary.items():
        text = re.sub(r'\b'+re.escape(original)+r'\b', new, text, flags=re.I)
    return text  

# Definimos una función que elimine stopwords
def stopwords_remover(text:str, stop_words_set:set):
    for word in stop_words_set:
        text = re.sub(r'\b'+re.escape(word)+r'\b', '', text, flags=re.I)
    return text 

# Definimos una función genérica para la limpieza del texto 
def dataframe_text_cleaner(input_df:pd.DataFrame | pd.Series, *dicts, stopwords:set|None=None):
    """
    This function takes a dataframe and applies text cleaning techniques. 
    You can pass additional dictionaries as additional positional arguments to replace words in the text of the dataframe.
    This applies the text correction for all the dataframe. Only considers str objects, everything else is ignored. 
    """
    # Lower case
    output_df = input_df.map(lambda x: x.lower() if type(x)==str else x)

    # Remove links 
    output_df = output_df.map(lambda x: re.sub(r'http[\w\.\/:]+','',x) if type(x)==str else x) 

    # Correct space typos, example: word.word -> word word 
    output_df = output_df.map(lambda x: re.sub(r'(?<=\w)[\.;,:-](?=\w)',' ',x) if type(x)==str else x) 

    # Normalize laugh 
    output_df = output_df.map(lambda x: re.sub(r'[\w]*(?:ja){2,}[\w]*','haha',x , flags=re.I) if type(x)==str else x)
    output_df = output_df.map(lambda x: re.sub(r'[\w]*(?:ha){2,}[\w]*','haha',x , flags=re.I) if type(x)==str else x)

    # Normalize the excesive use of vocals, ex: heeeeey -> hey
    output_df = output_df.map(lambda x: re.sub(r'a{2,}','a',x , flags=re.I) if type(x)==str else x)
    output_df = output_df.map(lambda x: re.sub(r'e{2,}','e',x , flags=re.I) if type(x)==str else x)
    output_df = output_df.map(lambda x: re.sub(r'i{2,}','i',x , flags=re.I) if type(x)==str else x)
    output_df = output_df.map(lambda x: re.sub(r'o{2,}','o',x , flags=re.I) if type(x)==str else x)
    output_df = output_df.map(lambda x: re.sub(r'u{2,}','u',x , flags=re.I) if type(x)==str else x)

    # Normalize and remove user handles, ex: @Bob -> USER
    output_df = output_df.map(lambda x: re.sub(r'@\w+','USER',x , flags=re.I) if type(x)==str else x)

    # Apply cleaning dictionaries 
    for dict in dicts:
        output_df = output_df.map(lambda x: dict_replacer(text=x, dictionary=dict) if type(x)==str else x)

    # Remove punctuation signs
    output_df = output_df.map(lambda x: re.sub(r'[\(\)\{\}—\|\'!¡¿\?:,;\.\/\+\*\-“”\"・]+','',x) if type(x)==str else x) 

    # Remove double spaces
    output_df = output_df.map(lambda x: re.sub(r'[ ]{2,}',' ',x) if type(x)==str else x) 

    # Fix hashtags separated from their word, ex: # hashtag -> #hashtag 
    output_df = output_df.map(lambda x: re.sub(r'(# )(?=\w)','#',x) if type(x)==str else x)

    # Remove stopwords  
    if stopwords != None:
        output_df = output_df.map(lambda x: stopwords_remover(x, stopwords) if type(x)==str else x)

    return output_df

# Definimos la función que transforma la data para el análisis de frecuencias 
def procesamiento_frecuencias (
        data_df: pd.DataFrame,
        taxonomy_df: pd.DataFrame,
        filter_mask: pd.Series | None = None,
        data_id_vars: list = ['Mes','NPS','Serial number'],
        taxo_id_vars: list = ['VOC'],
        taxo_cols_selection: list = ['VOC', 'DRIVERS', 'LBGUPS'],
        data_datetime_col: str = 'Mes'
) -> tuple: # Contains flat df and stacked df 

    output_df_flat = data_df[filter_mask].copy(deep=True)

    # Primer apilamiento, para quedarnos con las respuestas de netos en una sola columna 
    output_df_flat = pd.melt(
        output_df_flat,
        id_vars = data_id_vars,
        value_vars = data_df.filter(like='LBGUPS').columns.to_list(),
        var_name = 'Origin_variable',
        value_name = 'Responses'
    )

    output_df_flat.dropna(inplace=True)

    # Separar las respuestas en columnas 
    output_df_stacked = pd.merge(
        left=output_df_flat,
        right=output_df_flat['Responses'].str.split(pat=',', expand=True),
        left_index=True,
        right_index=True
    )

    # Apilar las columnas
    output_df_stacked = output_df_stacked.melt(
        id_vars = [*data_id_vars, 'Responses'],
        value_vars = output_df_stacked.filter(regex=r'\d').columns.to_list(),
        var_name='Answer No',
        value_name='Attributes'
        )

    # Fix de algunos temas de capitalizacion en netos / subnetos 
    output_df_stacked['Attributes'] = output_df_stacked['Attributes'].str.title()

    # Añadimos grupos 
    output_df_stacked = pd.merge(
        left = output_df_stacked,
        right = taxonomy_df[taxo_cols_selection],
        left_on = 'Attributes',
        right_on = taxo_id_vars,
        how = 'left'
        
    )

    # Quitamos columnas innecesarias
    output_df_stacked.drop(columns=['Answer No', 'VOC'], inplace=True)

    # Añadir columna por Qs y Hs
    output_df_flat['Qs'] = output_df_flat[data_datetime_col].dt.to_period('Q')
    output_df_flat['Hs'] = output_df_flat[data_datetime_col].apply(lambda x: f'{x.year} H{1 if x.month <= 6 else 2}')
    output_df_stacked['Qs'] =    output_df_stacked[data_datetime_col].dt.to_period('Q')
    output_df_stacked['Hs'] =    output_df_stacked[data_datetime_col].apply(lambda x: f'{x.year} H{1 if x.month <= 6 else 2}')

    return (output_df_flat, output_df_stacked)

# Definimos la función que toma la data original, la data stacked de frecuencias y devuelve la data procesada de verbatos
def procesamiento_verbatos (
        data_df: pd.DataFrame,
        stacked_freq_df: pd.DataFrame,
        data_filter_mask: pd.Series | None = None,
        time_filter_period: str | None = None,
        time_filter_var: str | None = None,
        data_id_vars: list = ['Mes','NPS','Serial number']
) -> pd.DataFrame:
    
    output_verbatim_stacked = data_df[data_filter_mask].copy(deep=True)
    
    # Primer apilamiento sobre verbatos 
    output_verbatim_stacked = pd.melt(
        frame = output_verbatim_stacked, 
        id_vars = data_id_vars, 
        value_vars = output_verbatim_stacked.filter(like='Verbatims').columns.to_list(),
        var_name = 'Origin_variable',
        value_name = 'Responses'
    )

    # Quitar los vacíos 
    output_verbatim_stacked.dropna(inplace=True)

    # Unir con el dataset stacked de drivers y lbgups
    output_verbatim_stacked = pd.merge(
        left=stacked_freq_df,
        right=output_verbatim_stacked[[*data_id_vars, 'Responses']],
        on=data_id_vars,
        how='left',
        suffixes=[None, '_verbatims']
    )

    # Limpiar los labels de los atributos 
    output_verbatim_stacked['Attributes'] = output_verbatim_stacked['Attributes'].apply(
        lambda x: re.sub(
            pattern=r'(\/ )|(Subneto )|(Neto )|(^\s)|(\/)',
            repl='',
            string=x,
            flags=re.I
            )
            if type(x) == str else x
        )

    # Aplica filtro de tiempo si esta correctamente especificado, de lo contrario solo quita NA's
    if (time_filter_period != None) & (time_filter_var != None):
        output_verbatim_stacked = output_verbatim_stacked[(output_verbatim_stacked[time_filter_var] == time_filter_period) & output_verbatim_stacked['Attributes'].notna()]
    else:
        output_verbatim_stacked = output_verbatim_stacked[output_verbatim_stacked['Attributes'].notna()]

    output_verbatim_stacked['Responses_verbatims_clean'] = dataframe_text_cleaner(output_verbatim_stacked['Responses_verbatims'], dict_contracciones, dict_slang, stopwords=spa_stopwords)

    return output_verbatim_stacked

# Generador de las nubes de palabras
def cloud_generator (df_dict:dict, column_name_text:str, column_name_categories:str|None = None, color_map:dict|None = None) -> None:
    """
    This wordcloud generator creates a worcloud image for each dataframe in the df_dict object. 
    Optionally we can color the words with a color map for certain categories. If no dictionary is passed, no specific color will be applied to the wordcloud.
    This function will save the images in the local dir ./Word Clouds/
    The name of the image will be the key value of the df_dict
    """
    # Recolor based on category frequency (simplified: majority category color)
    def color_func(word, font_size, position, orientation, random_state=None, **kwargs):
        # find rows containing the word
        mask = texts.str.contains(word, case=False, na=False)
        if mask.any():
            cat = categories[mask].mode()[0]
            return color_map.get(cat, 'gray')
        return 'gray'
    
    for df_key in df_dict.keys():
        
        # Get text 
        texts = df_dict[df_key][column_name_text]
        
        # Combine text 
        full_text = " ".join(str(t) for t in texts if pd.notna(t))

        # Create wordcloud
        wc = WordCloud(width=1200, height=800, background_color='white').generate(full_text)

        if (column_name_categories != None) & (color_map != None):
            categories = df_dict[df_key][column_name_categories] 
            wc_recolored = wc.recolor(color_func=color_func)

            plt.figure(figsize=(15,10))
            plt.imshow(wc_recolored, interpolation='bilinear')
            plt.axis('off')
            plt.savefig(f'Word Clouds/{df_key}.png')
        
        else: 
            plt.figure(figsize=(15,10))
            plt.imshow(wc, interpolation='bilinear')
            plt.axis('off')
            plt.savefig(f'Word Clouds/{df_key}.png')

# Ingesta de datos 

In [ ]:
DATABASE_PATH = 'Base analytics VOC Historico - Golden.xlsx'

taxonomy = pd.read_excel(DATABASE_PATH, sheet_name='2025 Taxo for freq')

# Fix de algunos temas de capitalizacion en netos / subnetos 
taxonomy['VOC'] = taxonomy['VOC'].str.title()
taxonomy.drop_duplicates(inplace=True)

taxonomy_pre = taxonomy[taxonomy['QClasificacion'] == 2].copy()
taxonomy_pos = taxonomy[taxonomy['QClasificacion'] == 1].copy()

data_pos = pd.read_excel(DATABASE_PATH, sheet_name='Postpago')
data_pre = pd.read_excel(DATABASE_PATH, sheet_name='Prepago')

### Filtro

In [ ]:
## HABILITA SOLO UNO ##

# # Filter mask Promotores
# filter_mask_pos = data_pos['NPS'] == 'Promotor'
# filter_mask_pre = data_pre['NPS'] == 'Promotor'

# # Filter mask Detractores
# filter_mask_pos = data_pos['NPS'] == 'Detractor'
# filter_mask_pre = data_pre['NPS'] == 'Detractor'

# # Filter mask Pasivos
# filter_mask_pos = data_pos['NPS'] == 'Pasivo'
# filter_mask_pre = data_pre['NPS'] == 'Pasivo'

# Filter mask All 
filter_mask_pos = data_pos['NPS'].notna()
filter_mask_pre = data_pre['NPS'].notna()

# Análisis frecuencias 

In [ ]:
# Aplicamos el preprocesamiento a las bases de datos 
freq_data_pos_flat, freq_data_pos_stacked = procesamiento_frecuencias(data_pos, taxonomy_pos, filter_mask_pos)
freq_data_pre_flat, freq_data_pre_stacked = procesamiento_frecuencias(data_pre, taxonomy_pre, filter_mask_pre)

# Creamos diccionario de DFs para exportar a Excel 
freq_output_dfs = {}

# DFs Pospago
freq_output_dfs['POS_BASE_Qs_Counts'] = freq_data_pos_flat['Qs'].value_counts()
freq_output_dfs['POS_BASE_Hs_Counts'] = freq_data_pos_flat['Hs'].value_counts()
freq_output_dfs['POS_LBGUPS_Qs_Counts'] = pd.crosstab(freq_data_pos_stacked['LBGUPS'], freq_data_pos_stacked['Qs'])
freq_output_dfs['POS_LBGUPS_Qs_Perc'] = pd.crosstab(freq_data_pos_stacked['LBGUPS'], freq_data_pos_stacked['Qs']) / freq_data_pos_flat['Qs'].value_counts()
freq_output_dfs['POS_DRIVERS_Qs_Counts'] = pd.crosstab(freq_data_pos_stacked['DRIVERS'], freq_data_pos_stacked['Qs'])
freq_output_dfs['POS_DRIVERS_Qs_Perc'] = pd.crosstab(freq_data_pos_stacked['DRIVERS'], freq_data_pos_stacked['Qs'])/ freq_data_pos_flat['Qs'].value_counts()
freq_output_dfs['POS_DRIVERS_Hs_Counts'] = pd.crosstab(freq_data_pos_stacked['DRIVERS'], freq_data_pos_stacked['Hs'])
freq_output_dfs['POS_DRIVERS_Hs_Perc'] = pd.crosstab(freq_data_pos_stacked['DRIVERS'], freq_data_pos_stacked['Hs'])/ freq_data_pos_flat['Hs'].value_counts()
freq_output_dfs['POS_ATTRIB_Hs_Counts'] = pd.crosstab(freq_data_pos_stacked['Attributes'], freq_data_pos_stacked['Hs'])
freq_output_dfs['POS_ATTRIB_Hs_Perc'] = pd.crosstab(freq_data_pos_stacked['Attributes'], freq_data_pos_stacked['Hs'])/ freq_data_pos_flat['Hs'].value_counts()

# DFs Prepago 
freq_output_dfs['PRE_BASE_Qs_Counts'] = freq_data_pre_flat['Qs'].value_counts()
freq_output_dfs['PRE_BASE_Hs_Counts'] = freq_data_pre_flat['Hs'].value_counts()
freq_output_dfs['PRE_LBGUPS_Qs_Counts'] = pd.crosstab(freq_data_pre_stacked['LBGUPS'], freq_data_pre_stacked['Qs'])
freq_output_dfs['PRE_LBGUPS_Qs_Perc'] = pd.crosstab(freq_data_pre_stacked['LBGUPS'], freq_data_pre_stacked['Qs']) / freq_data_pre_flat['Qs'].value_counts()
freq_output_dfs['PRE_DRIVERS_Qs_Counts'] = pd.crosstab(freq_data_pre_stacked['DRIVERS'], freq_data_pre_stacked['Qs'])
freq_output_dfs['PRE_DRIVERS_Qs_Perc'] = pd.crosstab(freq_data_pre_stacked['DRIVERS'], freq_data_pre_stacked['Qs'])/ freq_data_pre_flat['Qs'].value_counts()
freq_output_dfs['PRE_DRIVERS_Hs_Counts'] = pd.crosstab(freq_data_pre_stacked['DRIVERS'], freq_data_pre_stacked['Hs'])
freq_output_dfs['PRE_DRIVERS_Hs_Perc'] = pd.crosstab(freq_data_pre_stacked['DRIVERS'], freq_data_pre_stacked['Hs'])/ freq_data_pre_flat['Hs'].value_counts()
freq_output_dfs['PRE_ATTRIB_Hs_Counts'] = pd.crosstab(freq_data_pre_stacked['Attributes'], freq_data_pre_stacked['Hs'])
freq_output_dfs['PRE_ATTRIB_Hs_Perc'] = pd.crosstab(freq_data_pre_stacked['Attributes'], freq_data_pre_stacked['Hs'])/ freq_data_pre_flat['Hs'].value_counts()

# Output en Excel 
with pd.ExcelWriter('Analisis Frecuencias Output.xlsx') as writer:

    index_df = pd.DataFrame(freq_output_dfs.keys(), columns=['Sheet'])
    index_df['Link'] = index_df['Sheet'].apply(lambda x: f'=HYPERLINK("#\'{x}\'!A1", "Go to {x}")')
    index_df.to_excel(writer, sheet_name='Index')

    for df in freq_output_dfs.keys():
        freq_output_dfs[df].to_excel(writer, sheet_name=df)

# Verbatos 

In [ ]:
# Aplicamos el preprocesamiento a las bases de datos para la data de frecuencias (ocuparemos estos frames para extender el df stackeado para incluir verbatims)
_, freq_data_pos_stacked = procesamiento_frecuencias(data_pos, taxonomy_pos, filter_mask_pos)
_, freq_data_pre_stacked = procesamiento_frecuencias(data_pre, taxonomy_pre, filter_mask_pre)

# Procesamos y limpiamos la data de verbatims 
verbatims_data_pos_stacked = procesamiento_verbatos(data_pos, freq_data_pos_stacked, filter_mask_pos, '2025 H2', 'Hs')
verbatims_data_pre_stacked = procesamiento_verbatos(data_pre, freq_data_pre_stacked, filter_mask_pre, '2025 H2', 'Hs')

# Acomodo en frames por atributo
# Columnas a utilizar
verbatims_columns_list = ['NPS','Responses_verbatims_clean']

# Creamos diccionario de DFs para exportar a Excel 
verbatims_output_dfs = {}

# Acomodo en diccionario
for attribute in verbatims_data_pos_stacked['Attributes'].unique():
    verbatims_output_dfs[f'POS_{attribute}'] = verbatims_data_pos_stacked[verbatims_data_pos_stacked['Attributes']==attribute][verbatims_columns_list]

for attribute in verbatims_data_pre_stacked['Attributes'].unique():
    verbatims_output_dfs[f'PRE_{attribute}'] = verbatims_data_pre_stacked[verbatims_data_pre_stacked['Attributes']==attribute][verbatims_columns_list]

# Output en Excel 
with pd.ExcelWriter('Verbatims Output.xlsx') as writer:

    index_df = pd.DataFrame(verbatims_output_dfs.keys(), columns=['Sheet'])
    index_df['Link'] = index_df['Sheet'].apply(lambda x: f'=HYPERLINK("#\'{x}\'!A1", "Go to {x}")')
    index_df.to_excel(writer, sheet_name='Index')

    for df in verbatims_output_dfs.keys():
        verbatims_output_dfs[df].to_excel(writer, sheet_name=df[:31])

In [ ]:
with open('verbatims_output_dfs.pkl', 'wb') as pickle_file:
    pickle.dump(verbatims_output_dfs, pickle_file)

In [ ]:
with open('verbatims_output_dfs.pkl', 'rb') as pickle_file:
    verbatims_output_dfs = pickle.load(pickle_file)

# Nubes de palabras 

In [ ]:
nps_color_map = {'Promotor':'green','Pasivo':'orange','Detractor':'red'}

cloud_generator(
    df_dict = verbatims_output_dfs,
    column_name_text = 'Responses_verbatims_clean',
    column_name_categories = 'NPS',
    color_map = nps_color_map
)